# Step 3: 단기/장기 메모리의 결합 및 시맨틱 검색 (실전 레벨)

이 예제는 프로덕션(상용) 환경을 가정한 가장 진보된 아키텍처입니다.
1. **단기 메모리(Checkpointer) 결합**: 대화의 흐름(단기 메모리)을 유지하면서 장기 메모리를 동시에 활용합니다. 단기 메모리는 `thread_id`로 관리됩니다.
2. **임베딩 기반 시맨틱 검색**: 기억 데이터가 수만 개 쌓여도, 현재 대화 문맥과 가장 관련성 높은 기억만 골라서 의미(Semantic) 기반으로 찾아옵니다.
3. **카테고리 네임스페이스**: 기억을 8개의 서랍장(카테고리)으로 나누어 체계적으로 보관합니다.

In [8]:
from dotenv import load_dotenv
load_dotenv()

True

### 1. 임베딩이 적용된 장기 메모리 저장소(Store) 준비

In [9]:
from dataclasses import dataclass
from langchain.embeddings import init_embeddings
from langgraph.store.memory import InMemoryStore
import uuid
from datetime import datetime

@dataclass
class Context:
    user_id: str

# 임베딩 모델 초기화 (텍스트 기억을 수치화된 벡터로 변환하기 위함)
embeddings = init_embeddings("google_genai:gemini-embedding-001")

# InMemoryStore 생성 시 index 설정을 넘겨주면, 자동으로 시맨틱(벡터) 검색 기능이 활성화됩니다.
store = InMemoryStore(
    index={
        "embed": embeddings,
        "dims": 1536,
    }
)

### 2. 세분화된 메모리 제어 도구(Tools) 정의

In [10]:
from langchain.tools import ToolRuntime, tool

@tool
def get_user_info(runtime: ToolRuntime[Context]) -> str:
    """사용자의 기본 정보를 조회합니다."""
    assert runtime.store is not None
    user_info = runtime.store.get(("users",), runtime.context.user_id)
    return str(user_info.value) if user_info else "알 수 없는 사용자"

@tool
def save_user_info(
    preferences: list[str] = None,
    interests: list[str] = None,
    experiences: list[str] = None,
    current_activities: list[str] = None,
    goals: list[str] = None,
    routines: list[str] = None,
    concerns: list[str] = None,
    achievements: list[str] = None,
    runtime: ToolRuntime[Context] = None
) -> str:
    """
    사용자의 다양한 정보를 8개의 세분화된 카테고리(네임스페이스)로 분류하여 체계적으로 저장합니다.
    """
    assert runtime.store is not None
    store = runtime.store
    user_id = runtime.context.user_id
    current_time = datetime.now().isoformat()

    categories = {
        "preferences": preferences, "interests": interests, "experiences": experiences,
        "current_activities": current_activities, "goals": goals, "routines": routines,
        "concerns": concerns, "achievements": achievements
    }

    update_summary = []
    for category, values in categories.items():
        if values:
            for value in values:
                item_id = str(uuid.uuid4())
                # 네임스페이스를 (user_id, category)로 세분화하여 저장합니다.
                store.put(
                    (user_id, category),
                    item_id,
                    {"text": value, "created_at": current_time, "category": category}
                )
            update_summary.append(f"{len(values)}개의 {category}")

    return f"사용자 장기 기억 성공적 저장: {', '.join(update_summary)}"

In [11]:
@tool
def search_user_memories(
    query: str,
    category: str = None,
    limit: int = 5,
    runtime: ToolRuntime[Context] = None
) -> str:
    """
    사용자의 메모리를 '자연어 쿼리(query)'를 이용해 시맨틱(의미) 검색합니다.
    """
    assert runtime.store is not None
    store = runtime.store
    user_id = runtime.context.user_id

    # 핵심: Store에 임베딩 설정(index)이 되어 있으므로, query 파라미터만 넘기면 랭체인이 알아서 유사도 검색(Semantic Search)을 수행합니다.
    if category:
        namespace = (user_id, category)
        results = store.search(namespace, query=query, limit=limit)
        if not results:
            return f"{category} 카테고리에서 관련된 메모리를 찾을 수 없습니다."
        result_text = f"{category} 관련 메모리:\n"
        for item in results:
            result_text += f"- {item.value['text']} (저장일자: {item.value['created_at']})\n"
        return result_text
    else:
        categories = ["preferences", "interests", "experiences", "current_activities",
                     "goals", "routines", "concerns", "achievements"]
        all_results = []
        for cat in categories:
            try:
                results = store.search((user_id, cat), query=query, limit=limit)
                all_results.extend([(r, cat) for r in results])
            except:
                continue

        # 임베딩 유사도 검색을 통해 반환된 score 값으로 내림차순 정렬하여 가장 관련성 높은 기억만 추출합니다.
        all_results.sort(key=lambda x: x[0].score if hasattr(x[0], 'score') else 0, reverse=True)
        all_results = all_results[:limit]

        if not all_results:
            return "관련된 메모리를 찾을 수 없습니다."

        result_text = "관련 메모리:\n"
        for item, cat in all_results:
            result_text += f"[{cat}] {item.value['text']} (저장일자: {item.value['created_at']})\n"
        return result_text

### 3. 완벽한 에이전트 생성: 장기 메모리(`Store`) + 단기 메모리(`Checkpointer`) 결합

In [12]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3.1-flash-lite")

# 방금 전 대화(단기 메모리)를 기억하게 해주는 Checkpointer 생성
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_user_info, save_user_info, search_user_memories],
    store=store,                 # 영구 보관용 장기 메모리(Store) 연결
    checkpointer=checkpointer,   # 대화 흐름 유지용 단기 메모리(Checkpointer) 연결
    context_schema=Context,
    system_prompt="""당신은 누적된 사용자 메모리를 활용하여 맞춤 조언을 제공하는 라이프 코치입니다."""
)

### 4. 실행 루프 (단기/장기 동시 작동)
`thread_id`는 단기 메모리를, `user_id`는 장기 메모리를 식별하는 열쇠가 됩니다.

In [13]:
# 지속적인 대화를 통해 챗봇이 스스로 학습하고 기억을 활용하는지 테스트해보세요.
# 예) "내 이름은 일남이야" -> "나는 커피를 좋아해" -> "내 이름이 뭐게?" -> "내가 좋아하는 건 뭐지?"
while True:
    user_input = input("User: ")
    if user_input.lower() in ["q", "exit", "quit"]:
        break

    # config의 thread_id로 단기 메모리(대화 흐름) 관리, context로 장기 메모리(유저 기억) 유저 식별
    response = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config={"configurable": {"thread_id": "1"}}, # 단기 기억 대화 세션
        context=Context(user_id="user_123") # 장기 기억
    )

    for msg in response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

내 이름은 김일남이야
================================== Ai Message ==================================

[{'type': 'text', 'text': '반갑습니다, 김일남 님! 라이프 코치로서 앞으로 일남 님의 더 나은 일상과 성장을 위해 곁에서 돕게 되어 기쁩니다.\n\n본격적으로 일남 님에게 딱 맞는 조언을 드리기 위해, 지금 어떤 고민이 있으신지, 혹은 요즘 어떤 일에 관심이 있으신지 편하게 말씀해 주시겠어요? 작은 일상부터 장기적인 목표까지 무엇이든 좋습니다.', 'extras': {'signature': 'EjQKMgEMOdbHaQt4GdvNgDU9FS7IRgQ+38u/VU5xQJ4GDJCzJio4cb1YjgH1JJWD0d4+SxPA'}}]
================================ Human Message =================================

내 이름은 김일남이야
================================== Ai Message ==================================

[{'type': 'text', 'text': '반갑습니다, 김일남 님! 라이프 코치로서 앞으로 일남 님의 더 나은 일상과 성장을 위해 곁에서 돕게 되어 기쁩니다.\n\n본격적으로 일남 님에게 딱 맞는 조언을 드리기 위해, 지금 어떤 고민이 있으신지, 혹은 요즘 어떤 일에 관심이 있으신지 편하게 말씀해 주시겠어요? 작은 일상부터 장기적인 목표까지 무엇이든 좋습니다.', 'extras': {'signature': 'EjQKMgEMOdbHaQt4GdvNgDU9FS7IRgQ+38u/VU5xQJ4GDJCzJio4cb1YjgH1JJWD0d4+SxPA'}}]
============================

In [14]:
# 대화 세션이 변경된 경우를 가정하고 테스트
while True:
    user_input = input("User: ")
    if user_input.lower() in ["q", "exit", "quit"]:
        break

    response = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config={"configurable": {"thread_id": "2"}}, # 대화 세션 변경
        context=Context(user_id="user_123")
    )

    for msg in response["messages"]:
        msg.pretty_print()

================================ Human Message =================================

내가 좋아하는 커피는?
================================== Ai Message ==================================

[]
Tool Calls:
  search_user_memories (mFCa8lGT)
 Call ID: mFCa8lGT
  Args:
    query: 내가 좋아하는 커피
================================= Tool Message =================================
Name: search_user_memories

관련 메모리:
[preferences] 아이스 아메리카노(아아)를 좋아함 (저장일자: 2026-06-04T13:13:25.895188)

================================== Ai Message ==================================

[{'type': 'text', 'text': '메모리에 따르면, 당신은 **아이스 아메리카노(아아)**를 좋아하시는군요! 날씨가 더워지는 요즘 같은 때에 딱 어울리는 선택이네요. 오늘도 시원한 아아 한 잔과 함께 기분 좋은 하루 보내고 계신가요?', 'extras': {'signature': 'EjQKMgEMOdbHvNDUKT7xR7TGs8UfRH+kKXpcXoJnBVGwgcnwsmyfmEvvse7twrLk3DTNqbaD'}}]
================================ Human Message =================================

내가 좋아하는 커피는?
================================== Ai Message ==================================

[]
Tool Calls:
  search_user_memories